# Multi-Broker MQTT Subscriber

This notebook demonstrates subscribing to messages from local and public MQTT brokers.

## Setup

To use multiple brokers, edit `config.yaml` and change the `mqtt` section:

```yaml
mqtt:
  active_profiles: [local, mqtthq]  # or [local] or [mqtthq]
  profiles:
    local:
      host: "127.0.0.1"
      port: 1883
      tls: false
    mqtthq:
      host: "broker.mqttdashboard.com"
      port: 1883
      tls: false
```

## Imports

In [ ]:
import json
import time
from datetime import datetime
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector

## Load Configuration

In [ ]:
# Load configuration
cfg = load_config()

# Primary broker (first in active_profiles)
print(f"Primary broker: {cfg.mqtt.host}:{cfg.mqtt.port}")

# All available brokers
print(f"\nAvailable brokers:")
for profile_name, mqtt_cfg in cfg.mqtt_configs.items():
    print(f"  - {profile_name}: {mqtt_cfg.host}:{mqtt_cfg.port}")

---

# Option 1: Subscribe to Primary Broker Only

Simple approach for subscribing to the primary broker. Works with both single and multiple broker configs.

In [ ]:
# Storage for received messages
messages_received = []

def on_message_primary(client, userdata, msg):
    """Callback when a message arrives."""
    try:
        payload = json.loads(msg.payload.decode())
        message_info = {
            "timestamp": datetime.now().isoformat(),
            "topic": msg.topic,
            "payload": payload
        }
        messages_received.append(message_info)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Received on {msg.topic}: {payload}")
    except json.JSONDecodeError:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Received non-JSON on {msg.topic}: {msg.payload.decode()}")

# Create connector for primary broker
connector = MqttConnector(cfg.mqtt, client_id_suffix="subscriber-primary")
connector.client.on_message = on_message_primary
connector.connect()

# Wait for connection and subscribe
if connector.wait_for_connection(timeout=5.0):
    # Subscribe to all messages under the base topic
    topic_filter = f"{cfg.mqtt.base_topic}/#"
    connector.client.subscribe(topic_filter)
    print(f"✓ Connected and subscribed to {topic_filter}")
    print(f"  Broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
    print(f"\nListening for messages... (run publisher cells or send manually)")
else:
    print(f"✗ Failed to connect to {cfg.mqtt.host}:{cfg.mqtt.port}")

In [ ]:
# Wait and collect messages for 10 seconds
print(f"\nListening for 10 seconds...")
time.sleep(10)

if messages_received:
    print(f"\nReceived {len(messages_received)} message(s):")
    for i, msg in enumerate(messages_received, 1):
        print(f"\n{i}. {msg['timestamp']}")
        print(f"   Topic: {msg['topic']}")
        print(f"   Payload: {msg['payload']}")
else:
    print("No messages received yet. Try running the publisher notebook!")

In [ ]:
# Clean up
connector.disconnect()
print("\nDisconnected from primary broker")

---

# Option 2: Subscribe to a Specific Named Broker

Choose which broker to subscribe from by name. This works when using multiple brokers. to se the name of the availabul broksers see the output of the load configuation cell

In [ ]:
# Pick a specific broker by name
broker_name = "hivemq_cloud"  # use "local" for local broker and "hivemq_cloud" to use the public broker


# Storage for messages from this broker
messages_from_broker = []

def on_message_named(client, userdata, msg):
    """Callback for named broker subscription."""
    try:
        payload = json.loads(msg.payload.decode())
        message_info = {
            "timestamp": datetime.now().isoformat(),
            "topic": msg.topic,
            "payload": payload
        }
        messages_from_broker.append(message_info)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {broker_name}: {msg.topic} = {payload}")
    except json.JSONDecodeError:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {broker_name}: {msg.topic} = {msg.payload.decode()}")

if broker_name not in cfg.mqtt_configs:
    print(f"✗ Broker '{broker_name}' not available. Available: {list(cfg.mqtt_configs.keys())}")
else:
    broker_cfg = cfg.mqtt_configs[broker_name]
    
    # Create connector for selected broker
    connector = MqttConnector(broker_cfg, client_id_suffix=f"subscriber-{broker_name}")
    connector.client.on_message = on_message_named
    connector.connect()
    
    if connector.wait_for_connection(timeout=5.0):
        topic_filter = f"{broker_cfg.base_topic}/#"
        connector.client.subscribe(topic_filter)
        print(f"✓ Connected and subscribed to {broker_name}")
        print(f"  Host: {broker_cfg.host}:{broker_cfg.port}")
        print(f"  Topic: {topic_filter}")
        print(f"\nListening for messages... (run publisher cells or send manually)")
    else:
        print(f"✗ Failed to connect to {broker_cfg.host}:{broker_cfg.port}")

In [ ]:
# Wait and collect messages for 10 seconds
print(f"\nListening for 10 seconds...")
time.sleep(10)

if messages_from_broker:
    print(f"\nReceived {len(messages_from_broker)} message(s) from {broker_name}:")
    for i, msg in enumerate(messages_from_broker, 1):
        print(f"\n{i}. {msg['timestamp']}")
        print(f"   Topic: {msg['topic']}")
        print(f"   Payload: {msg['payload']}")
else:
    print(f"No messages received from {broker_name}. Try running the publisher notebook!")

In [ ]:
# Clean up
connector.disconnect()
print(f"\nDisconnected from {broker_name}")

---

# Option 3: Subscribe to Multiple Brokers Simultaneously

Listen for messages on all active brokers at the same time.

In [ ]:
# Storage for all messages by broker
all_messages = {}

def make_message_handler(profile_name):
    """Create a message handler for a specific broker."""
    def on_message(client, userdata, msg):
        try:
            payload = json.loads(msg.payload.decode())
            message_info = {
                "timestamp": datetime.now().isoformat(),
                "topic": msg.topic,
                "payload": payload
            }
            all_messages[profile_name].append(message_info)
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {profile_name}: {msg.topic}")
        except json.JSONDecodeError:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {profile_name}: {msg.topic} (non-JSON)")
    return on_message

# Create and connect to all available brokers
multi_connectors = {}
for profile_name, broker_cfg in cfg.mqtt_configs.items():
    all_messages[profile_name] = []
    
    connector = MqttConnector(broker_cfg, client_id_suffix=f"subscriber-multi-{profile_name}")
    connector.client.on_message = make_message_handler(profile_name)
    connector.connect()
    
    if connector.wait_for_connection(timeout=5.0):
        multi_connectors[profile_name] = connector
        topic_filter = f"{broker_cfg.base_topic}/#"
        connector.client.subscribe(topic_filter)
        print(f"✓ {profile_name}: subscribed to {topic_filter}")
    else:
        print(f"✗ {profile_name}: failed to connect")

print(f"\nListening on {len(multi_connectors)} broker(s)...")

In [ ]:
# Wait and collect messages for 10 seconds
print(f"Listening for 10 seconds...")
time.sleep(10)

print(f"\n=== Summary ===")
total_messages = 0

for profile_name, messages in all_messages.items():
    count = len(messages)
    total_messages += count
    print(f"\n{profile_name}: {count} message(s)")
    
    for i, msg in enumerate(messages, 1):
        print(f"  {i}. [{msg['timestamp'].split('T')[1]}] {msg['topic']}")

if total_messages == 0:
    print("\nNo messages received yet.")
    print("Tips:")
    print("  1. Run the publisher notebook to send messages")
    print("  2. Make sure your brokers have matching topic prefixes")
    print("  3. Check that your brokers are running and accessible")

In [ ]:
# Clean up: disconnect from all brokers
for profile_name, connector in multi_connectors.items():
    connector.disconnect()
    print(f"Disconnected from {profile_name}")

print("\nAll connections closed")